In [1]:
#Loading the dataset using pandas

import pandas as pd

df = pd.read_csv('content_engagement.csv')

In [6]:
import numpy as np
from urllib.parse import urlparse

In [2]:
df.head()

,url,title,author,author_page,subtitle,claps,responses,reading_time,tag,date
0,https://medium.com/@dharmeshpanchmatia/data-an...,Data Analytics and AI/ML platform for eCommerce,dharmeshpanchmatia,https://medium.com/@dharmeshpanchmatia,Improve user pr,30.0,0.0,5.0,Big Data,2021-01-01
1,https://medium.com/predict/the-journey-from-an...,The journey from an Apple Silicon M1 powered l...,dimitrisv,https://medium.com/@dimitrisv,NaN,42.0,0.0,5.0,Big Data,2021-01-01
2,https://medium.com/@rajsaraogi/demystifying-kp...,Demystifying KPIs in Analytics,rajsaraogi,https://medium.com/@rajsaraogi,NaN,19.0,0.0,3.0,Big Data,2021-01-01
3,https://medium.com/@bigdataschool/%D0%B1%D0%BE...,Большая разница: чем структурированная потоков...,bigdataschool,https://medium.com/@bigdataschool,NaN,0.0,0.0,6.0,Big Data,2021-01-01
4,https://medium.com/@arnowa-44509/iot-a-new-way...,IoT: A New Way of Life,arnowa-44509,https://medium.com/@arnowa-44509,"IoT is more than just the Internet of things, ...",0.0,0.0,3.0,Big Data,2021-01-01


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47660 entries, 0 to 47659
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   url           47660 non-null  object 
 1   title         47660 non-null  object 
 2   author        47660 non-null  object 
 3   author_page   47660 non-null  object 
 4   subtitle      13728 non-null  object 
 5   claps         47660 non-null  float64
 6   responses     47660 non-null  float64
 7   reading_time  47660 non-null  float64
 8   tag           47660 non-null  object 
 9   date          47660 non-null  object 
dtypes: float64(3), object(7)
memory usage: 3.6+ MB


In [4]:
df.describe(include="all")

,url,title,author,author_page,subtitle,claps,responses,reading_time,tag,date
count,47660,47660,47660,47660,13728,47660.000000,47660.000000,47660.000000,47660,47660
unique,47659,46746,22570,23812,13318,NaN,NaN,NaN,7,351
top,https://medium.com/@veronikabelokhvostova/im-a...,Artificial Intelligence,everythingforfootball,https://medium.com/@everythingforfootball,Introduction,NaN,NaN,NaN,Data Science,2021-10-28
freq,2,19,315,315,43,NaN,NaN,NaN,11415,355
mean,NaN,NaN,NaN,NaN,NaN,47.194125,0.243265,5.133928,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,227.351856,1.506246,3.758287,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,3.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,4.000000,0.000000,4.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,42.000000,0.000000,6.000000,NaN,NaN


In [5]:
#checking for the presence of missing data aur null values in the dataset 
df.isnull().sum()

url                 0
title               0
author              0
author_page         0
subtitle        33932
claps               0
responses           0
reading_time        0
tag                 0
date                0
dtype: int64

In [7]:
print(df.duplicated().sum(), df['url'].duplicated().sum())

0 1


In [8]:
df = df.drop_duplicates()

In [9]:
print(df.duplicated().sum(), df['url'].duplicated().sum())

0 1


In [10]:
df[df['url'].duplicated(keep=False)] 

,url,title,author,author_page,subtitle,claps,responses,reading_time,tag,date
24526,https://medium.com/@veronikabelokhvostova/im-a...,"<strong class=""markup--strong markup--h3-stron...",veronikabelokhvostova,https://medium.com/@veronikabelokhvostova,NaN,591.0,0.0,7.0,Data Science,2021-10-07
25593,https://medium.com/@veronikabelokhvostova/im-a...,Facebook destroyed my family andmy mother and ...,lexahe3185,https://medium.com/@lexahe3185,NaN,0.0,0.0,0.0,Data Science,2021-10-12


In [11]:
df = df.drop_duplicates(subset='url')

In [12]:
print(df.duplicated().sum(), df['url'].duplicated().sum())

0 0


In [18]:
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True).dt.tz_localize(None)
print(df['date'].dtype, df['date'].isna().sum())
df = df.dropna(subset=['date'])

datetime64[ns] 0


In [14]:
# ubtitle flag + text cleanup
df['has_subtitle'] = df['subtitle'].notna().astype(int)
df['subtitle'] = df['subtitle'].fillna('')
for c in ['title', 'subtitle']:
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True).str.strip()

In [15]:
# English titles only
ascii_share = df['title'].apply(lambda t: sum(ch.isascii() for ch in t) / max(len(t), 1))
print((ascii_share < 0.9).sum(), 'non-English rows dropped')
df = df[ascii_share >= 0.9].copy()

1523 non-English rows dropped


In [16]:
# Sanity check
print(df[['claps', 'responses', 'reading_time']].describe())
print(df['date'].min(), df['date'].max())

              claps     responses  reading_time
count  46136.000000  46136.000000  46136.000000
mean      47.963196      0.250499      5.114401
std      230.605113      1.529650      3.670214
min        0.000000      0.000000      0.000000
25%        0.000000      0.000000      3.000000
50%        4.000000      0.000000      4.000000
75%       44.000000      0.000000      6.000000
max    23000.000000    125.000000    131.000000
2021-01-01 2021-12-31


In [19]:
# Features
df['title_length'] = df['title'].str.len()
df['title_word_count'] = df['title'].str.split().str.len()
df['has_number'] = df['title'].str.contains(r'\d').astype(int)
df['is_question'] = df['title'].str.endswith('?').astype(int)
df['is_how_to'] = df['title'].str.lower().str.startswith('how to').astype(int)
df['subtitle_word_count'] = df['subtitle'].str.split().str.len()
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
df['length_bucket'] = pd.cut(df['reading_time'], [0, 3, 7, np.inf],
                             labels=['short', 'medium', 'long'],
                             include_lowest=True).astype(str)

In [20]:
def get_pub(u):
    p = urlparse(u)
    if p.netloc != 'medium.com':
        return p.netloc
    first = p.path.strip('/').split('/')[0]
    return 'personal' if first.startswith('@') else first

df['publication'] = df['url'].apply(get_pub)
df['author_article_count'] = df.groupby('author')['url'].transform('nunique')
df['log_claps'] = np.log1p(df['claps'])
p90 = df.groupby('tag')['claps'].transform(lambda s: s.quantile(0.9))
df['high_impact'] = ((df['claps'] >= p90) & (df['claps'] > 0)).astype(int)

In [22]:
print((df['reading_time'] == 0).sum())          #no of rows having 0 mins reading time 
df = df[df['reading_time'] > 0].copy()
df['has_response'] = (df['responses'] > 0).astype(int)

285


In [26]:
df = df.drop(columns=['author_page'], errors='ignore').rename(columns={'date': 'publish_date'})
df['publish_date'] = pd.to_datetime(df['publish_date']).dt.date
df.to_csv('content_engagement_clean.csv', index=False, encoding='utf-8')
df.to_parquet('content_engagement_clean.parquet', index=False)

In [27]:
print(df.shape, list(df.columns))

(45851, 25) ['url', 'title', 'author', 'subtitle', 'claps', 'responses', 'reading_time', 'tag', 'publish_date', 'has_subtitle', 'title_length', 'title_word_count', 'has_number', 'is_question', 'is_how_to', 'subtitle_word_count', 'day_of_week', 'month', 'is_weekend', 'length_bucket', 'publication', 'author_article_count', 'log_claps', 'high_impact', 'has_response']


In [29]:
df = df.rename(columns={'month': 'publish_month'})
df[['claps', 'responses']] = df[['claps', 'responses']].astype('int64')
int_cols = df.select_dtypes('integer').columns
df[int_cols] = df[int_cols].astype('int64')
print(df.dtypes)
df.to_parquet('content_engagement_clean.parquet', index=False)
df.to_csv('content_engagement_clean.csv', index=False, encoding='utf-8')

url                      object
title                    object
author                   object
subtitle                 object
claps                     int64
responses                 int64
reading_time            float64
tag                      object
publish_date             object
has_subtitle              int64
title_length              int64
title_word_count          int64
has_number                int64
is_question               int64
is_how_to                 int64
subtitle_word_count       int64
day_of_week              object
publish_month             int64
is_weekend                int64
length_bucket            object
publication              object
author_article_count      int64
log_claps               float64
high_impact               int64
has_response              int64
dtype: object
